# 01 — Sanity visuals

Checks the core idea of the project visually:

1. **Toy graphs** — a cyclic triangle stays a loop (H1 bar never dies); a one-way (transitive) triangle is filled (no bar); a back-and-forth pair is a loop; a ring that a fourth account "fills" dies.
2. **Synthetic data** — planted wash rings get low (suspicious) edge weights.
3. **One candidate ring** — the ring inside its neighbourhood, and the persistence diagram of that neighbourhood.

Kernel: select the project's `.venv`. All logic lives in `src/ringtda`; this notebook only calls it and plots.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from ringtda.data import toy
from ringtda.data.synthetic import SyntheticConfig, generate
from ringtda.graph.windows import aggregate_pairs
from ringtda.graph.weights import edge_weights, to_digraph
from ringtda.topology.candidates import candidate_rings, neighborhood
from ringtda.topology.persistence import available_backends, directed_flag_persistence
from ringtda.viz import draw_digraph, plot_diagram

print("persistence backends available:", available_backends())

## 1. Toy graphs

Top row: the graph (edge labels are filtration weights). Bottom row: its persistence diagram. Pink triangles are H1 (loops); a point on the dashed line never dies.

In [ ]:
examples = {
    "cyclic A→B→C→A": toy.cyclic_triangle(0.1),
    "transitive A→B, B→C, A→C": toy.transitive_triangle(0.1),
    "reciprocal A⇄B": toy.reciprocal_pair(0.1),
    "ring filled via account 3": toy.shortcut_filled_triangle(0.1, 0.6),
}
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for col, (name, G) in enumerate(examples.items()):
    draw_digraph(axes[0, col], G, title=name)
    plot_diagram(axes[1, col], directed_flag_persistence(G), max_value=1.0)
    bars = directed_flag_persistence(G)[1]
    axes[1, col].set_title(f"H1 bars: {bars.round(2).tolist()}", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Synthetic data: do planted rings get low weights?

Lower weight = more suspicious. Wash-ring edges should sit far left of background traffic.

In [ ]:
tx, planted = generate(SyntheticConfig(seed=0))
pairs = edge_weights(aggregate_pairs(tx))
G = to_digraph(pairs)
print(f"{len(tx):,} transactions, {len(pairs):,} account pairs, {len(planted)} planted structures")

kind = {}
for p in planted:
    for e in p.edges:
        kind[e] = p.pattern
pairs["kind"] = [kind.get((s, d), "background") for s, d in zip(pairs.src, pairs.dst)]

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 41)
for k, c in [("background", "#adb5bd"), ("transitive_decoy", "#fab005"), ("wash_ring", "#d6336c")]:
    ax.hist(pairs.loc[pairs.kind == k, "weight"], bins=bins, color=c, alpha=0.75, label=k, density=True)
ax.axvline(pairs.weight.quantile(0.10), color="k", ls="--", lw=1, label="lowest 10% cutoff")
ax.set_xlabel("edge weight (low = suspicious)")
ax.set_ylabel("density")
ax.legend()
plt.show()

## 3. Candidate rings and one ring's neighbourhood

In [ ]:
tau = pairs.weight.quantile(0.10)
res = candidate_rings(G, tau=tau)
print(f"{len(res.candidates)} candidate rings from {res.n_sccs} strongly connected components; truncated SCCs: {res.truncated_sccs}")

found = {frozenset(c.edges) for c in res.candidates}
rings = [p for p in planted if p.pattern == "wash_ring"]
print(f"planted rings recovered: {sum(frozenset(p.edges) in found for p in rings)}/{len(rings)}")

ring = next(p for p in rings if len(p.accounts) == 3)
H = neighborhood(G, ring.accounts, hops=1)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
draw_digraph(axes[0], H, highlight_edges=ring.edges, title=f"planted 3-ring {ring.accounts} + 1-hop neighbourhood")
plot_diagram(axes[1], directed_flag_persistence(H), title="persistence of the neighbourhood")
plt.tight_layout()
plt.show()

## What to look for

- Section 1: only the *transitive* triangle has no H1 point; the filled ring dies at 0.6.
- Section 2: pink (wash ring) mass is left of the dashed line.
- Section 3: all planted rings recovered; the ring's loop appears as an H1 point born early (low weight).

Save any figure you want for the report with `fig.savefig("../results/figures/<name>.png", dpi=200)`.